# 02 - Cleaning, Filtering, and Feature Engineering

Filters to reliable segments, then builds every feature behind the Speed Safety Score: `speed_gap`, `road_mismatch`, `urban_flag`, `vru_exposure`, `bio_risk`, `confidence_weight`, and `mapillary_url`.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import geopandas as gpd

from src.utils import load_geojson, detect_country, harmonize_schema, filter_reliable, load_helmet_layer
from src.features import engineer_features

pd.set_option('display.max_columns', None)

FILES = [
    '../data/raw/ADB_Innovation_Thailand.geojson',
    '../data/raw/ADB_Innovation_Maharashtra.geojson',
]

## Filter to reliable segments

`ExcludeFromSpeedSPI == 0` (where present) AND `AnalysisStatus == 'Valid'` (where present) AND `Sample_Size_Total >= 1000`.

In [2]:
reliable_frames, low_conf_frames = [], []
for path in FILES:
    gdf = load_geojson(path)
    country = detect_country(gdf)
    gdf = harmonize_schema(gdf, country)
    reliable, low_conf = filter_reliable(gdf)
    print(f'{path}: total={len(gdf)} reliable={len(reliable)} low_confidence={len(low_conf)}')
    reliable_frames.append(reliable)
    low_conf_frames.append(low_conf)

reliable = gpd.GeoDataFrame(pd.concat(reliable_frames, ignore_index=True), geometry='geometry', crs='EPSG:4326')
low_confidence = gpd.GeoDataFrame(pd.concat(low_conf_frames, ignore_index=True), geometry='geometry', crs='EPSG:4326')
low_confidence['low_confidence'] = True

reliable.to_file('../data/processed/segments_reliable.geojson', driver='GeoJSON')
low_confidence.to_file('../data/processed/segments_low_confidence.geojson', driver='GeoJSON')
print(f'\nSaved segments_reliable.geojson ({len(reliable)}) and segments_low_confidence.geojson ({len(low_confidence)})')

../data/raw/ADB_Innovation_Thailand.geojson: total=55884 reliable=11524 low_confidence=44360
../data/raw/ADB_Innovation_Maharashtra.geojson: total=14082 reliable=3022 low_confidence=11060

Saved segments_reliable.geojson (14546) and segments_low_confidence.geojson (55420)


## Pedestrian exposure data (WorldPop)

`vru_exposure` also uses a population-density proxy sampled from [WorldPop](https://hub.worldpop.org/project/list) 100m gridded population rasters (CC BY 4.0, no API key required) — the most direct available signal for "how many people are near this road," complementing the `urban_flag`/helmet-wearing proxies already used.

**One-time manual setup** (not automated — these files are too large to fetch or commit automatically): download the constrained population-density GeoTIFF for each country from WorldPop's [Population Density](https://hub.worldpop.org/geodata/listing?id=76) listing (2020, 100m resolution) and place them at:

- `../data/external/worldpop/ind_pop_density_2020.tif` (India)
- `../data/external/worldpop/tha_pop_density_2020.tif` (Thailand)

If these files aren't present, the cell below prints a 0% coverage warning and the rest of the pipeline still runs — `vru_exposure` degrades gracefully to the original urban_flag + helmet-wearing formula (see `src/features.py:compute_vru_exposure`).

In [3]:
from src.pedestrian_exposure import compute_pedestrian_exposure

reliable = compute_pedestrian_exposure(reliable)
coverage = reliable.groupby('country')['pop_density_raw'].apply(lambda s: s.notna().mean() * 100)
print('pop_density_raw coverage by country (% of segments with a sampled value):')
print(coverage.round(1).to_string())
if (coverage == 0).any():
    print('WARNING: 0% coverage for at least one country -- WorldPop rasters not found under data/external/worldpop/. ' 'vru_exposure will fall back to the urban_flag + helmet-wearing formula for those segments.')

pop_density_raw coverage by country (% of segments with a sampled value):
country
India       0.0
Thailand    0.0


## Feature engineering

`vru_exposure` blends three signals: `urban_flag`, a region-level low-helmet-compliance risk signal (spatially joined from `Archive/*.gpkg`, 4 zones for Maharashtra vs 77 provinces for Thailand), and a WorldPop population-density proxy sampled just above. Full-strength weights are 0.24 / 0.36 / 0.40 respectively — chosen so that segments without WorldPop coverage renormalise back to the original 0.40 urban_flag / 0.60 helmet-risk formula exactly (see `src/features.py:VRU_EXPOSURE_WEIGHTS` and `_blend`).

In [4]:
helmet_layers = {
    'India': load_helmet_layer(
        '../Archive/Road_Safety_Performance_Indicators__Maharashtra_(Feature).gpkg',
        'Boundaries_4helmet', 'India',
    ),
    'Thailand': load_helmet_layer(
        '../Archive/Road_Safety_Performance_Indicators__Thailand_(Feature).gpkg',
        'Thailand_Province_Boundaries', 'Thailand',
    ),
}

reliable = engineer_features(reliable, helmet_layers=helmet_layers,
                              pop_density_norm=reliable['pop_density_norm'])
display(reliable[['segment_id', 'country', 'road_class', 'speed_gap', 'speed_gap_norm',
                   'road_mismatch', 'urban_flag', 'pop_density_raw', 'pop_density_norm',
                   'vru_exposure', 'recommended_speed_limit',
                   'speed_limit_gap', 'bio_risk', 'confidence_weight', 'mapillary_url']].head(5))

,segment_id,country,road_class,speed_gap,speed_gap_norm,road_mismatch,urban_flag,pop_density_raw,pop_density_norm,vru_exposure,recommended_speed_limit,speed_limit_gap,bio_risk,confidence_weight,mapillary_url
0,2,Thailand,primary,49.200000,0.559091,0.000000,0.0,NaN,NaN,0.137061,70.0,-4.0,0.137061,1.0,https://www.mapillary.com/app/?lat=14.90433086...
1,6,Thailand,primary,14.666667,0.166667,0.000000,1.0,NaN,NaN,0.537061,60.0,10.0,0.537061,1.0,https://www.mapillary.com/app/?lat=14.93997637...
2,9,Thailand,primary,7.250000,0.082386,0.285714,0.0,NaN,NaN,0.137061,70.0,20.0,0.137061,1.0,https://www.mapillary.com/app/?lat=14.90423914...
3,18,Thailand,secondary,0.000000,0.000000,0.333333,1.0,NaN,NaN,0.406709,50.0,30.0,0.406709,1.0,https://www.mapillary.com/app/?lat=13.88443570...
4,19,Thailand,secondary,0.000000,0.000000,0.333333,1.0,NaN,NaN,0.406709,50.0,30.0,0.406709,1.0,https://www.mapillary.com/app/?lat=13.90434435...


### Sanity check: secondary road posted at 55 km/h

The brief's note that 55 km/h is "5 km/h above" the 60 km/h Safe System max for secondary roads is arithmetically incorrect (55 < 60). `road_mismatch` is only positive when the *posted limit exceeds* the class ceiling, so this case correctly scores 0.

In [5]:
sec55 = reliable[(reliable['road_class'] == 'secondary') & (reliable['SpeedLimit'] == 55)]
print(f"Secondary roads posted at 55 km/h: {len(sec55)} segments.")
print(f"road_mismatch values: {sec55['road_mismatch'].unique()} (0.0 is correct here)")

Secondary roads posted at 55 km/h: 696 segments.
road_mismatch values: [0.] (0.0 is correct here)


In [6]:
reliable.to_parquet('../data/processed/_reliable_features.parquet')
low_confidence.to_parquet('../data/processed/_low_confidence.parquet')
print('Saved intermediate parquet files for notebook 03.')

Saved intermediate parquet files for notebook 03.
